In [0]:
staging_bkp_cash_path=dbutils.widgets.get("staging_bkp_cash_path")
bearsadjustments=dbutils.widgets.get("bearsadjustments")
office_path = dbutils.widgets.get("office_path")
client_path=dbutils.widgets.get("client_path")
date_path=dbutils.widgets.get("date_path")
payerdimension_path=dbutils.widgets.get("payerdimension_path")
adjustmentcode_path=dbutils.widgets.get("adjustmentcode_path")
cubeserviceofficetxnweekendingdate_path=dbutils.widgets.get("cubeserviceofficetxnweekendingdate_path")
mart_fact_adjustments_path=dbutils.widgets.get("mart_fact_adjustments_path")
mart_adjustments=dbutils.widgets.get("mart_adjustments")

In [0]:
count = spark.sql(f"SELECT COUNT(*) as cnt FROM {mart_fact_adjustments_path}").collect()[0]['cnt']
if count == 0 :
    print(f"Full load for {mart_fact_adjustments_path} executed")
    spark.sql(f"""
INSERT INTO {mart_fact_adjustments_path}
(
  reporting_week_ending_date_key,
  posted_date_key,
  source_system_key,
  office_key,
  client_key,
  payor_key,
  adjustment_code_key,
  adjustment_type_key,
  invoice_number,
  adjustment_amount
)
SELECT
  CAST(Reporting_Week_Ending_Date_Key AS BIGINT),
  CAST(Posted_Date_Key AS BIGINT),
  CAST(Source_System_Key AS INT),
  CAST(Office_Key AS INT),
  CAST(Client_Key AS INT),
  CAST(Payor_Key AS INT),
  CAST(Adjustment_Code_Key AS INT),
  CAST(Adjment_Type_Key AS INT),
  CAST(Invoice_Number AS STRING),
  CAST(Adjustment_Amount AS DECIMAL(18,2))
FROM {mart_adjustments}
WHERE Source_System_Key IN (0, 19);

    """)

In [0]:
spark.sql(
    f""" 
DROP VIEW IF EXISTS BAdjustment;
 """
)

spark.sql(
    f""" 
CREATE OR REPLACE TEMPORARY VIEW BAdjustment AS
SELECT 
  CASE dayofweek(PostedDate)
    WHEN 1 THEN PostedDate  -- Sunday
    WHEN 2 THEN date_add(PostedDate, -1)  
    WHEN 3 THEN date_add(PostedDate, -2)  -- Tuesday
    WHEN 4 THEN date_add(PostedDate, -3)  -- Wednesday
    WHEN 5 THEN date_add(PostedDate, 3)   -- Thursday
    WHEN 6 THEN date_add(PostedDate, 2)   -- Friday
    WHEN 7 THEN date_add(PostedDate, 1)   -- Saturday
  END AS ReportingWeekendingDate,
  'BEARS' AS SourceSystem,
  PayorTypeCode,
  OfficeNumber,
  CASE WHEN Product LIKE 'R%' THEN 0 ELSE SUM(Applied) END AS AdjustedAmount,
  PayorID,
  BillToName,
  ClientNumber,
  PostedDate,
  DepositDate,
  BatchID,
  CheckID,
  InvoiceNumber,
  Type,
  BatchNumber,
  Bank,
  Product
FROM {staging_bkp_cash_path} C
WHERE AppliedOnMR = 0 AND Applied <> 0
GROUP BY 
  CASE dayofweek(PostedDate)
    WHEN 1 THEN PostedDate
    WHEN 2 THEN date_add(PostedDate, -1)
    WHEN 3 THEN date_add(PostedDate, -2)
    WHEN 4 THEN date_add(PostedDate, -3)
    WHEN 5 THEN date_add(PostedDate, 3)
    WHEN 6 THEN date_add(PostedDate, 2)
    WHEN 7 THEN date_add(PostedDate, 1)
  END,
  PayorTypeCode,
  OfficeNumber,
  PayorID,
  BillToName,
  ClientNumber,
  PostedDate,
  DepositDate,
  BatchID,
  CheckID,
  InvoiceNumber,
  Type,
  BatchNumber,
  Bank,
  Product;

   """
)

In [0]:
spark.sql(
    f"""
    TRUNCATE TABLE {bearsadjustments};
    """
)

In [0]:
spark.sql(
    f"""
INSERT INTO {bearsadjustments} (
  reportingweekendingdatekey,
  sourcesystemkey,
  adjustmentposteddatekey,
  posteddatetkey,
  depositdatekey,
  officekey,
  invoicenumber,
  clientkey,
  payorkey,
  adjustment_code_key,
  adjustmentamount
)
SELECT 
  W.WeekEndingDateKey AS reportingweekendingdatekey,
  0 AS sourcesystemkey,
  CAST(NULL AS INTEGER) AS adjustmentposteddatekey,
  PD.DateKey AS posteddatetkey,
  DD.DateKey AS depositdatekey,
  O.OfficeKey AS officekey,
  V.InvoiceNumber AS invoicenumber,
  C.ClientKey AS clientkey,
  P.PayerKey AS payorkey,
  A.Adjustment_Code_Key AS adjustment_code_key,
  V.AdjustedAmount AS adjustmentamount
FROM BAdjustment V
JOIN {office_path} O 
  ON O.OfficeNumber = V.OfficeNumber
LEFT JOIN {client_path} C 
  ON C.SourceSystemId = V.ClientNumber 
  AND C.OfficeNumber = O.OfficeNumber 
  AND C.SourceSystem = 'BEARS'
LEFT JOIN {date_path} PD 
  ON PD.CalendarDate = V.PostedDate
LEFT JOIN {date_path} DD 
  ON DD.CalendarDate = V.DepositDate
LEFT JOIN {payerdimension_path} P 
  ON P.PayerID = V.PayorID
LEFT JOIN {adjustmentcode_path} A 
  ON (A.Adjustment_Code || ' ' || '-' || ' ' || A.Adjustment_Code_Description) = V.Product
LEFT JOIN {cubeserviceofficetxnweekendingdate_path} W 
  ON W.WeekEndingDate = V.ReportingWeekendingDate;
    """
)

In [0]:
spark.sql(
    f"""
    INSERT INTO {mart_fact_adjustments_path} (
        reporting_week_ending_date_key,
        posted_date_key,
        source_system_key,
        office_key,
        client_key,
        payor_key,
        adjustment_code_key,
        adjustment_type_key,
        invoice_number,
        adjustment_amount
    )
    SELECT 
        reportingweekendingdatekey       AS reporting_week_ending_date_key,
        posteddatetkey                   AS posted_date_key,
        sourcesystemkey                  AS source_system_key,
        officekey                        AS office_key,
        clientkey                        AS client_key,
        payorkey                         AS payor_key,
        adjustment_code_key              AS adjustment_code_key,
        NULL                             AS adjustment_type_key,
        invoicenumber                    AS invoice_number,
        CAST(adjustmentamount AS DECIMAL(18,2)) AS adjustment_amount
    FROM {bearsadjustments};
    """
)
